# Week 8 - Deep Learning for Representations and Dynamics: from PCA to Autoencoders, CNNs, and SHRED

Deep learning can feel like a discontinuity in the course -- as though everything
we built with the singular value decomposition were suddenly replaced by an
opaque network of weights. This lesson makes the opposite case, then shows where
the networks genuinely go beyond the SVD. The simplest neural network that learns
a representation, a **linear autoencoder** trained to minimize reconstruction
error, is not a rival to PCA: it *is* PCA. The Eckart-Young theorem guarantees
that the best rank-$k$ approximation of a matrix is its truncated SVD, so gradient
descent walks a linear autoencoder straight to the principal subspace we already
know how to compute in closed form.

That equivalence is the *floor*. The rest of the lesson relaxes the "linear"
assumption and measures what it buys. A **nonlinear autoencoder** bends its latent
surface to follow a curved manifold that no flat plane can capture; and **SHRED**
(a shallow recurrent decoder) reconstructs an entire spatiotemporal field of excitable tissue from
a handful of sensor **electrodes** -- and, more honestly, pins down *when* its nonlinearity is
needed: it ties a linear decoder on an organized (periodic) rhythm but decisively beats it on a
chaotic, fibrillating one. We train these with **PyTorch**, device-agnostic: the code uses a GPU
when one is available (as on Colab) and falls back to the CPU otherwise, and the
models are deliberately small so each trains in seconds on either device. A small
**convolutional network** on the Week-5a blood cells then closes the loop, learning image
features that beat a linear pixel model -- deep learning's answer to the eigen-cells
introduced in Week 5a.

Reinforcement learning -- learning to *act* under feedback, the canonical case being how to dose a drug -- is a different paradigm from the representation learning in this lesson, so it lives with the **capstone**: a runnable [reinforcement-learning preview](../capstone_rl_preview.ipynb) opens the capstone section, and the project develops it fully on real warfarin PK/PD data. The genuinely large sequence models (transformers and foundation-model embeddings for clinical text and biological sequences) sit beyond this course's scope; we note where they extend the same ideas.

**Reading.** Kutz, *Data-Driven Modeling & Scientific Computation*, 2nd ed. --
Chapter 15, sections 10--11 (autoencoders and the shallow recurrent decoder, SHRED, as
nonlinear extensions of the SVD) and Chapter 6 (neural networks as curve fits); Chapter 20,
section 5 extends SHRED to sensing and spatiotemporal fields. (Reinforcement learning, Chapter 19, is previewed and taken up in the capstone; transformers and foundation models, Chapter 22, sit beyond this course's scope.)
Read those for the architectures and their derivations; everything below is developed in
our own terms and validated against our own fixtures.

**Learning goals.**

- State and *demonstrate numerically* the equivalence between a linear
  autoencoder trained by gradient descent and PCA / the truncated SVD.
- Read a training-loss curve and confirm it descends to the closed-form
  Eckart-Young optimum rather than beating it.
- Apply the same linear-representation machinery to a real biomedical feature
  matrix and interpret the resulting two-dimensional latent space.
- Train a **nonlinear autoencoder** in PyTorch and show it recovers a curved
  manifold that the best linear latent cannot.
- Use **SHRED** to reconstruct a tissue's full electrical field from a few electrodes, and
  pin down *when* the nonlinearity is required -- on a chaotic (fibrillating) field, not an
  organized (periodic) one.
- Train a small **convolutional neural network (CNN)** on BloodMNIST blood-cell images and
  show its learned features beat a raw-pixel linear baseline and Week 5a's linear eigen-cells.
- Close with an explicit interpretation (a claim and named limitations) -- honest that the
  nonlinear gains, unlike the linear=PCA equivalence, carry no closed-form
  guarantee.

## Setup

We seed every random number generator and apply the course plotting style so the
figures below are deterministic and reproducible from a cold kernel, and we select
a compute device (GPU if available, otherwise CPU).

```{admonition} Which paradigm?
:class: note
**Data-driven.** Autoencoders and SHRED sit at the far data-driven end of everything the course has built -- the inductive counterpart to Week 5a's eigen-cells. You hand the network measurements -- 30 morphological features of 569 breast-cancer tumors, a curved S-curve manifold, a ring of excitable tissue read by three of ninety-six electrodes -- and it learns coordinates and dynamics with no mechanism written down. That flexibility forfeits interpretability and, mostly, guarantees: the nonlinear autoencoder and the LSTM field-reconstructor are validated only empirically. The exception is the floor -- a *linear* autoencoder trained by gradient descent is provably PCA (Eckart-Young), the same subspace Week 5a computed in closed form. Reach here when the structure is real but you cannot yet name its mechanism; the capstone then braids this data-driven power back toward a model.
```


In [ ]:
# Colab setup: install the ddm4bio course library.
# No-op when ddm4bio is already importable (e.g. the course-site build), so
# this cell is safe everywhere. It is hidden from the rendered site via the
# "remove-cell" tag, but runs when this notebook is opened in Google Colab.
try:
    import ddm4bio  # noqa: F401
except ModuleNotFoundError:
    %pip install -q "ddm4bio @ git+https://github.com/symbiont-ai/ddm4bio.git" umap-learn
    import ddm4bio  # noqa: F401

In [ ]:
import numpy as np
import torch

import ddm4bio
from ddm4bio import seed_everything
from ddm4bio.viz.style import set_style

seed_everything()
torch.manual_seed(0)
set_style()

# Device-agnostic: use a GPU when one is available (e.g. Colab), else the CPU.
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print(f"ddm4bio version: {ddm4bio.__version__}")
print(f"PyTorch {torch.__version__} on device: {device}")

## 1. The equivalence, on synthetic ground truth

A **linear autoencoder** is the smallest network that learns a representation. It
compresses each centered input row $x \in \mathbb{R}^d$ to a $k$-dimensional code
$z = x W_e$ with an encoder matrix $W_e \in \mathbb{R}^{d \times k}$, then expands
it back with a decoder matrix $W_d \in \mathbb{R}^{k \times d}$ to a
reconstruction $\hat{x} = z W_d$. There are no nonlinear activations anywhere --
that omission is the whole point. Training minimizes the mean squared
reconstruction error $\frac{1}{N d}\sum_i \lVert \hat{x}_i - x_i \rVert^2$ by
gradient descent on the two weight matrices.

Here is the claim we will verify. Among *all* rank-$k$ linear maps, the
Eckart-Young theorem says the best squared-error reconstruction of a centered
data matrix is its truncated SVD -- exactly what PCA computes. A linear
autoencoder can only ever realize a rank-$k$ linear map, so the lowest loss it can
possibly reach is the PCA reconstruction error. Gradient descent, given enough
steps, converges to that floor. The encoder/decoder it lands on spans the **same
subspace** as the top-$k$ principal components (up to a rotation and scaling
inside the code, which the reconstruction is blind to).

To test this cleanly we build data whose true rank we control: 300 samples living
on a random 3-dimensional subspace inside 12-dimensional feature space, plus a thin
layer of isotropic noise. An honest rank-3 reconstruction should capture almost
everything.

In [ ]:
from ddm4bio.methods.decomposition import explained_variance_ratio, svd_lowrank
from ddm4bio.methods.validation import reconstruction_error

rng = np.random.default_rng(0)

n_samples, n_features, true_rank = 300, 12, 3

# Rank-3 generator: latent scores (300 x 3) times loadings (3 x 12), plus noise.
latent = rng.standard_normal((n_samples, true_rank))
loadings = rng.standard_normal((true_rank, n_features))
noise = 0.02 * rng.standard_normal((n_samples, n_features))
X = latent @ loadings + noise

# Center once; both PCA and the autoencoder operate on the centered matrix.
Xc = X - X.mean(axis=0, keepdims=True)

evr = explained_variance_ratio(X)
print(f"Data matrix X: {X.shape} (samples x features)")
print(f"Explained-variance ratio (first 6 components):\n{np.round(evr[:6], 4)}")
print(f"First {true_rank} components capture {evr[:true_rank].sum():.4%} of variance.")

### 1a. The PCA reconstruction (closed form)

The rank-$k$ PCA reconstruction is the truncated SVD of the centered matrix:
keep the top $k$ singular triplets and multiply them back together. This is the
Eckart-Young optimum -- no rank-$k$ linear map reconstructs `Xc` with lower
squared error.

In [ ]:
k = true_rank

U, s, Vt = svd_lowrank(Xc, k)
X_pca = U @ np.diag(s) @ Vt  # best rank-k reconstruction (Eckart-Young)

err_pca = reconstruction_error(Xc, X_pca, kind="rel_l2")
print(f"PCA (rank-{k}) relative-L2 reconstruction error: {err_pca:.6f}")

### 1b. The linear autoencoder (gradient descent)

Now the network. We initialize small random encoder and decoder matrices and run
plain full-batch gradient descent on the mean-squared reconstruction error. The
gradients are elementary -- no autodiff framework, no GPU, just two matrix
multiplies per step -- which is exactly why this fits in an offline notebook.

In [ ]:
d = n_features
N = Xc.shape[0]

# Small random initial weights (a fresh, seeded generator for reproducibility).
init_rng = np.random.default_rng(1)
W_e = 0.1 * init_rng.standard_normal((d, k))  # encoder: features -> code
W_d = 0.1 * init_rng.standard_normal((k, d))  # decoder: code -> features

lr = 0.05
n_epochs = 4000
losses = np.empty(n_epochs)

for epoch in range(n_epochs):
    Z = Xc @ W_e            # (N, k) latent codes
    X_hat = Z @ W_d         # (N, d) reconstruction
    R = X_hat - Xc          # residual
    losses[epoch] = np.mean(R**2)

    # Gradients of the mean-squared error w.r.t. each weight matrix.
    grad_W_d = (Z.T @ R) * (2.0 / (N * d))
    grad_W_e = (Xc.T @ (R @ W_d.T)) * (2.0 / (N * d))

    W_e -= lr * grad_W_e
    W_d -= lr * grad_W_d

X_ae = (Xc @ W_e) @ W_d
err_ae = reconstruction_error(Xc, X_ae, kind="rel_l2")
print(f"Autoencoder (rank-{k}) relative-L2 reconstruction error: {err_ae:.6f}")
print(f"Final training MSE: {losses[-1]:.3e}")

The training loss should fall and then flatten onto the PCA reconstruction error.
It cannot dip below it: the dashed line is the Eckart-Young floor, and the
network is asymptotically pinned to it.

In [ ]:
import matplotlib.pyplot as plt

pca_mse = np.mean((X_pca - Xc) ** 2)

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(losses, linewidth=1.5, label="autoencoder training MSE")
ax.axhline(pca_mse, linestyle="--", color="0.4",
           label=f"PCA (Eckart-Young) MSE = {pca_mse:.2e}")
ax.set_yscale("log")
ax.set_xlabel("Gradient-descent epoch")
ax.set_ylabel("Mean squared reconstruction error")
ax.set_title("Linear autoencoder descends to the PCA optimum")
ax.legend(loc="upper right")
fig;

### 1c. Do the two reconstructions actually match?

A loss curve landing on the right floor is suggestive; the decisive test is
whether the two methods reconstruct the *same data* the same way. We compare the
reconstructions directly, and -- more stringently -- measure the principal angles
between the subspace the autoencoder's decoder spans and the top-$k$ PCA
subspace. Angles near zero mean the network found the principal subspace itself,
not merely a comparable error.

In [ ]:
# Direct agreement of the two reconstructions.
recon_gap = reconstruction_error(X_pca, X_ae, kind="rel_l2")

# Subspace agreement: principal angles between decoder row-space and PCA components.
Q_ae, _ = np.linalg.qr(W_d.T)   # orthonormal basis for the autoencoder subspace
Q_pca, _ = np.linalg.qr(Vt.T)   # orthonormal basis for the top-k PCA subspace
cos_angles = np.clip(np.linalg.svd(Q_ae.T @ Q_pca, compute_uv=False), -1.0, 1.0)
principal_angles_deg = np.degrees(np.arccos(cos_angles))

print(f"Autoencoder-vs-PCA reconstruction difference (rel-L2): {recon_gap:.4f}")
print(f"Principal angles between subspaces (deg): "
      f"{np.round(principal_angles_deg, 3)}")

The reconstructions agree to a fraction of a percent and the principal angles are
essentially zero. That is the equivalence made concrete: a network trained only
to reconstruct its input, with no knowledge of eigenvectors, has rediscovered the
principal subspace. A scatter of one method's reconstruction of a single feature against the other's makes the point visually -- the points sit on the identity line.

In [ ]:
feat = 0  # inspect a single feature dimension across all samples

fig, ax = plt.subplots(figsize=(5.2, 5))
lo = min(X_pca[:, feat].min(), X_ae[:, feat].min())
hi = max(X_pca[:, feat].max(), X_ae[:, feat].max())
ax.plot([lo, hi], [lo, hi], linestyle="--", color="0.5", label="identity")
ax.scatter(X_pca[:, feat], X_ae[:, feat], s=14, alpha=0.6, label="samples")
ax.set_xlabel(f"PCA reconstruction (feature {feat})")
ax.set_ylabel(f"Autoencoder reconstruction (feature {feat})")
ax.set_title("Per-sample reconstructions coincide")
ax.legend(loc="upper left")
fig;

## 2. A biomedical latent space

The synthetic fixture proved the equivalence; now we point the same machinery at
a real biomedical measurement. The Wisconsin **breast-cancer** dataset bundled
with scikit-learn describes 569 tumor samples by 30 morphological features
computed from digitized fine-needle-aspirate images (cell radius, texture,
concavity, and so on), each labeled benign or malignant. This is the kind of
wide, correlated feature matrix where a low-dimensional latent is genuinely
useful: many of the 30 features are near-redundant descriptors of a few
underlying tumor properties.

Because the features live on wildly different numeric scales, we standardize each
to zero mean and unit variance first -- otherwise PCA (and the autoencoder) would
simply chase whichever feature happens to have the largest raw units.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()
X_raw = data.data                       # (569, 30) morphological features
y = data.target                         # 0 = malignant, 1 = benign

# Standardize features, then center for the decomposition.
X_std = (X_raw - X_raw.mean(axis=0)) / X_raw.std(axis=0)
X_bio = X_std - X_std.mean(axis=0, keepdims=True)

evr_bio = explained_variance_ratio(X_std)
print(f"Feature matrix: {X_raw.shape} (tumors x features)")
print(f"Class balance (malignant, benign): {np.bincount(y)}")
print(f"Top-2 components capture {evr_bio[:2].sum():.2%} of variance.")

We compress to a two-dimensional latent with both methods and confirm they agree
on real data too, then read the latent space. Unlike the synthetic fixture, no
two components capture *all* the variance here -- a real 30-feature tumor
description is not exactly rank 2 -- so the reconstruction error is substantial
and honest. The question is whether the two dimensions we keep are biologically
organized.

In [ ]:
k_bio = 2

# Closed-form PCA reconstruction.
U_b, s_b, Vt_b = svd_lowrank(X_bio, k_bio)
X_bio_pca = U_b @ np.diag(s_b) @ Vt_b
err_bio_pca = reconstruction_error(X_bio, X_bio_pca, kind="rel_l2")

# Linear autoencoder trained the same way as before.
d_b, N_b = X_bio.shape[1], X_bio.shape[0]
ae_rng = np.random.default_rng(2)
We_b = 0.1 * ae_rng.standard_normal((d_b, k_bio))
Wd_b = 0.1 * ae_rng.standard_normal((k_bio, d_b))
lr_b, epochs_b = 0.02, 8000
for _ in range(epochs_b):
    Zb = X_bio @ We_b
    Rb = Zb @ Wd_b - X_bio
    Wd_b -= lr_b * (Zb.T @ Rb) * (2.0 / (N_b * d_b))
    We_b -= lr_b * (X_bio.T @ (Rb @ Wd_b.T)) * (2.0 / (N_b * d_b))
X_bio_ae = (X_bio @ We_b) @ Wd_b
err_bio_ae = reconstruction_error(X_bio, X_bio_ae, kind="rel_l2")

print(f"PCA (k=2) reconstruction error:        {err_bio_pca:.4f}")
print(f"Autoencoder (k=2) reconstruction error: {err_bio_ae:.4f}")
print(f"Method-to-method difference (rel-L2):   "
      f"{reconstruction_error(X_bio_pca, X_bio_ae, kind='rel_l2'):.4f}")

The two latents reconstruct the tumors almost identically -- the equivalence is
not an artifact of the clean synthetic data. Now the biomedical payoff: we plot
each tumor by its two PCA latent scores and color by diagnosis. A linear latent
is only worth keeping if the biology organizes along it.

In [ ]:
from ddm4bio.methods.decomposition import pca_reduce

scores = pca_reduce(X_std, n_components=2)

fig, ax = plt.subplots(figsize=(7, 5.5))
for label, name in [(0, "malignant"), (1, "benign")]:
    sel = y == label
    ax.scatter(scores[sel, 0], scores[sel, 1], s=18, alpha=0.65, label=name)
ax.set_xlabel("Latent dimension 1")
ax.set_ylabel("Latent dimension 2")
ax.set_title("Two-dimensional linear latent of breast-cancer morphology")
ax.legend(title="diagnosis", loc="upper right")
fig;

## 3. Where nonlinearity earns its keep: a nonlinear autoencoder

The equivalence above is a *floor*, not a ceiling. A linear latent can only draw a
flat subspace through the data; when the real structure is a **curved manifold**,
no plane can capture it. Single-cell transcriptomes are the canonical case -- cells
along a differentiation trajectory trace a curve through gene-expression space --
but we can see the effect cleanly on a fixture whose truth we control: the
**S-curve**, a two-dimensional sheet folded into three dimensions.

We compress it to a two-dimensional latent two ways: the best linear plane (PCA),
and a small **nonlinear autoencoder** with the same encoder->latent->decoder shape
but `tanh` nonlinearities that let the latent surface bend to follow the fold.

In [ ]:
import torch.nn as nn
from sklearn.datasets import make_s_curve

# A 2-D sheet folded into 3-D: the intrinsic dimension is 2, but it is CURVED.
X_curve, position = make_s_curve(1500, noise=0.05, random_state=0)
X_curve = (X_curve - X_curve.mean(0)) / X_curve.std(0)

# Linear baseline: PCA to 2 dims is the best flat plane through the sheet.
X_centered = X_curve - X_curve.mean(0)
_, _, Vt = np.linalg.svd(X_centered, full_matrices=False)
pca_latent = X_centered @ Vt[:2].T
pca_recon = pca_latent @ Vt[:2] + X_curve.mean(0)
linear_err = np.linalg.norm(X_curve - pca_recon) / np.linalg.norm(X_curve)


class Autoencoder(nn.Module):
    '''Encoder -> latent -> decoder, with tanh nonlinearities between layers.'''

    def __init__(self, n_features, hidden, latent):
        super().__init__()
        self.encode = nn.Sequential(
            nn.Linear(n_features, hidden), nn.Tanh(), nn.Linear(hidden, latent)
        )
        self.decode = nn.Sequential(
            nn.Linear(latent, hidden), nn.Tanh(), nn.Linear(hidden, n_features)
        )

    def forward(self, x):
        return self.decode(self.encode(x))


torch.manual_seed(0)
X_curve_t = torch.tensor(X_curve, dtype=torch.float32, device=device)
autoencoder = Autoencoder(n_features=3, hidden=32, latent=2).to(device)
optimizer = torch.optim.Adam(autoencoder.parameters(), lr=1e-2)
for epoch in range(1500):
    optimizer.zero_grad()
    loss = ((autoencoder(X_curve_t) - X_curve_t) ** 2).mean()
    loss.backward()
    optimizer.step()

with torch.no_grad():
    ae_err = (
        torch.linalg.norm(autoencoder(X_curve_t) - X_curve_t)
        / torch.linalg.norm(X_curve_t)
    ).item()
    ae_latent = autoencoder.encode(X_curve_t).cpu().numpy()

print("Curved 2-D manifold in 3-D, compressed to a 2-D latent:")
print(f"  linear (PCA plane)    reconstruction rel-L2 = {linear_err:.3f}")
print(f"  nonlinear autoencoder reconstruction rel-L2 = {ae_err:.3f}")
print("  -> the nonlinear latent follows the fold; the flat plane cannot.")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), constrained_layout=True)
for ax, latent, title in [
    (axes[0], pca_latent, f"Linear (PCA) latent -- rel-L2 {linear_err:.2f}"),
    (axes[1], ae_latent, f"Nonlinear autoencoder latent -- rel-L2 {ae_err:.2f}"),
]:
    dots = ax.scatter(latent[:, 0], latent[:, 1], c=position, cmap="viridis", s=8)
    ax.set_xlabel("latent dimension 1")
    ax.set_ylabel("latent dimension 2")
    ax.set_title(title)
fig.colorbar(dots, ax=axes, label="position along the manifold", shrink=0.8)
fig;

The colour is the true position along the fold. The linear latent smears
those positions together -- a flat plane cannot separate points that the fold
brings close in 3-D -- while the nonlinear autoencoder lays them out as a smooth
gradient: it has *unrolled* the manifold, halving the reconstruction error at the
same latent dimension. That is the whole reason to reach for a nonlinear model:
not more parameters for their own sake, but a latent that can bend to the data.


### 3b. Parametric vs non-parametric: what a reusable function buys

The nonlinear autoencoder did something t-SNE and UMAP never do: it learned a *function*. Its
encoder maps any point in the data space to a latent code, and its decoder maps back -- so you
can feed it points it has never seen, read off their coordinates, reconstruct them, and score the
result with a real error in the original units. t-SNE and UMAP instead fit **one** embedding of
**one** dataset: they place the points you gave them and, for t-SNE, offer no way at all to
position a new one.


In [ ]:
import time
from sklearn.manifold import TSNE
import umap

# The AE learned a reusable MAP. Section 3 standardized X_curve in place, so recover the
# training mean/std, then encode+decode 300 genuinely UNSEEN S-curve points.
X_raw_train, _ = make_s_curve(1500, noise=0.05, random_state=0)
mu, sd = X_raw_train.mean(0), X_raw_train.std(0)
X_new_raw, position_new = make_s_curve(300, noise=0.05, random_state=1)
X_new = (X_new_raw - mu) / sd
X_new_t = torch.tensor(X_new, dtype=torch.float32, device=device)
t = time.perf_counter()
with torch.no_grad():
    recon_new = autoencoder.decode(autoencoder.encode(X_new_t)).cpu().numpy()
oos_ms = (time.perf_counter() - t) * 1e3
oos_err = np.linalg.norm(X_new - recon_new) / np.linalg.norm(X_new)
print(f"AE on 300 UNSEEN points: rel-L2 {oos_err:.2f} (train {ae_err:.2f}); "
      f"encode+decode took {oos_ms:.1f} ms")

# The embeddings, by contrast, fit ONE picture of ONE dataset.
tsne = TSNE(n_components=2, perplexity=30, init="pca", random_state=0)
emb_tsne = tsne.fit_transform(X_curve)
print(f"t-SNE : KL divergence {tsne.kl_divergence_:.2f}; a .transform for new points? "
      f"{hasattr(tsne, 'transform')}")
reducer = umap.UMAP(n_components=2, random_state=0)
emb_umap = reducer.fit_transform(X_curve)
print(f"UMAP  : .transform? {hasattr(reducer, 'transform')}; "
      f".inverse_transform? {hasattr(reducer, 'inverse_transform')}; "
      f"UMAP.transform(new) -> {reducer.transform(X_new).shape}")


In [ ]:
import matplotlib.pyplot as plt

panels = [
    (ae_latent, f"Nonlinear AE latent (PARAMETRIC)\nreconstruction rel-L2 {ae_err:.2f}"),
    (emb_tsne, f"t-SNE (non-parametric)\nKL {tsne.kl_divergence_:.2f}"),
    (emb_umap, "UMAP (non-parametric)\nrandom_state=0"),
]
fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.3), constrained_layout=True)
for ax, (emb, title) in zip(axes, panels):
    dots = ax.scatter(emb[:, 0], emb[:, 1], c=position, cmap="viridis", s=8)
    ax.set_title(title, fontsize=10); ax.set_xticks([]); ax.set_yticks([])
fig.colorbar(dots, ax=axes, label="true position along the manifold", shrink=0.8)
plt.show()


The contrast is concrete. The autoencoder is **parametric** -- a reusable, invertible map: it
placed 300 unseen points in a fraction of a millisecond (a single forward pass) and reconstructed
them to an error close to its training error, and its decoder *is* an explicit inverse. t-SNE is
**non-parametric** in the strict sense -- it has no `.transform` at all, so a new point cannot be
placed without refitting the whole embedding. Plain UMAP is a middle case: it does expose an
approximate, graph-based `.transform` (and even an `.inverse_transform`) against its frozen
training graph, but that is *not* a smooth learned map -- the genuine parametric bridge is
Parametric UMAP (Sainburg, McInnes & Gentner, 2021), a neural encoder trained to reproduce the
UMAP objective.

Two lessons for the paradigm thread. First, the far data-driven end is not uniform: the
autoencoder is data-driven yet **model-producing** -- a function you can carry to new patients or
cells -- while t-SNE/UMAP are data-driven and **purely descriptive**, a picture of the data you
already have. Second, if you look closely the UMAP panel can even *fragment a single connected
sheet* into pieces -- the same "apparent clusters from a continuum" warning Week 5b quantified,
here on a manifold we know is one unbroken fold. A reconstruction error in data units -- which
only the parametric map offers -- is the check that keeps you honest.


## 4. SHRED: reconstructing a whole field -- and when the nonlinearity earns its keep

Week 7 asked *how far ahead* you can forecast. Here is the complement: *how few
measurements* do you need to see the whole thing? **SHRED** -- a SHallow REcurrent
DEcoder -- places a handful of **sensors** (here, electrodes) at fixed locations, feeds
their short recent history to a small **LSTM**, and lets a decoder reconstruct the
*entire* spatial field at each moment.

SHRED is a recent, specialized architecture (Williams, Zahn & Kutz, 2024), not a
canonical primitive -- so the honest question is not "does it run?" but "when does its
**nonlinearity** actually beat a plain linear decoder?" That question has a real answer,
and it is the lesson. We put SHRED on two regimes of the *same* excitable tissue and
evaluate both **honestly**:

- **split by time, never at random** -- a random split of these overlapping 40-step
  windows would *leak*: neighbours share 39 of 40 lagged steps, so near-duplicate
  contexts would sit on both sides. We hold out a contiguous *later* block instead.
- against a **fair, ridge-regularized linear** decoder over the same history (an
  unregularized least-squares baseline just overfits and flatters SHRED).

The two regimes are **organized reentry** -- a single depolarization wave circulating
*periodically* -- and **fibrillation** -- the wave broken into wavelets that drift
*chaotically*. Watch what changes.

In [ ]:
import torch.nn as nn
from sklearn.linear_model import Ridge


def fhn_reentry(n_time=1200, n_space=96):
    """Organized rhythm: one depolarization wave circulating a ring of tissue -- periodic."""
    u = -1.2 * np.ones(n_space); w = -0.6 * np.ones(n_space); head = n_space // 12
    u[:head] = 2.0; w[head:2 * head] = 1.5
    field = np.empty((n_time, n_space))
    for t in range(n_time):
        for _ in range(40):
            lap = np.roll(u, 1) - 2 * u + np.roll(u, -1)
            u = u + 0.02 * (1.2 * lap + u - u**3 / 3 - w)
            w = w + 0.02 * 0.02 * (u + 0.7 - 0.8 * w)
        field[t] = u
    return field


def fibrillation(n_time=3000, n_space=96):
    """Disorganized rhythm: two wavelets whose positions drift CHAOTICALLY (Lorenz-driven)."""
    def lorenz(n, dt=0.01, sub=8):
        s, r, b = 10.0, 28.0, 8 / 3; state = np.array([1.0, 1.0, 1.0]); out = []
        for _ in range(n):
            for _ in range(sub):
                x, y, z = state
                state = state + dt * np.array([s * (y - x), x * (r - z) - y, x * y - b * z])
            out.append(state.copy())
        return np.array(out)

    lat = lorenz(n_time + 600)[600:]                     # drop the transient
    grid = np.linspace(0, 1, n_space)
    bump = lambda c, a, wid: a[:, None] * np.exp(-((grid[None, :] - c[:, None])**2) / (2 * wid[:, None]**2))
    c1 = 0.30 + 0.22 * np.tanh(lat[:, 0] / 12)           # two chaotically drifting centres
    c2 = 0.70 + 0.22 * np.tanh(lat[:, 1] / 15)
    a1 = 0.7 + 0.3 * (0.5 + 0.5 * np.tanh(lat[:, 1] / 15))
    a2 = 0.7 + 0.3 * (0.5 + 0.5 * np.tanh(lat[:, 2] / 30))
    wid = 0.045 + 0.02 * (0.5 + 0.5 * np.tanh(lat[:, 2] / 30))
    return bump(c1, a1, wid) + bump(c2, a2, wid)


class Shred(nn.Module):
    """Shallow recurrent decoder: an LSTM over electrode history, then a field decoder."""

    def __init__(self, n_sensors, hidden, n_space):
        super().__init__()
        self.lstm = nn.LSTM(n_sensors, hidden, batch_first=True)
        self.decode = nn.Sequential(nn.Linear(hidden, 128), nn.ReLU(), nn.Linear(128, n_space))

    def forward(self, x):
        history, _ = self.lstm(x)
        return self.decode(history[:, -1])


LAGS = 40


def split_by_time(field, n_sensors, seed=0):
    """Random electrodes; lagged histories -> full field; split by TIME (train early, test late)."""
    field = (field - field.mean()) / field.std()
    n_time, n_space = field.shape
    rng = np.random.default_rng(seed)
    sensors = np.sort(rng.choice(n_space, n_sensors, replace=False))
    readings = field[:, sensors]
    sequences = np.stack([readings[i - LAGS:i] for i in range(LAGS, n_time)])
    targets = field[LAGS:]
    cut = int(0.75 * len(sequences))                     # gap of LAGS before the test block -> no leak
    return sequences, targets, np.arange(0, cut - LAGS), np.arange(cut, len(sequences)), sensors, n_space


def linear_reconstruct(features, targets, train_idx, test_idx, alpha=1.0):
    """Best *ridge-regularized* linear map from features to the full field; held-out rel-L2."""
    model = Ridge(alpha=alpha).fit(features[train_idx], targets[train_idx])
    pred = model.predict(features[test_idx])
    return np.linalg.norm(pred - targets[test_idx]) / np.linalg.norm(targets[test_idx])


def train_shred(sequences, targets, train_idx, test_idx, n_sensors, n_space, epochs=200, batch=256):
    """Train SHRED on the electrode histories; return held-out error and the reconstruction."""
    torch.manual_seed(0)
    model = Shred(n_sensors, 64, n_space).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=3e-3)
    to_t = lambda a: torch.tensor(a, dtype=torch.float32, device=device)
    x_train, y_train = to_t(sequences[train_idx]), to_t(targets[train_idx])
    x_test, y_test = to_t(sequences[test_idx]), to_t(targets[test_idx])
    gen = torch.Generator().manual_seed(0)
    for _ in range(epochs):
        for idx in torch.randperm(len(x_train), generator=gen).split(batch):
            optimizer.zero_grad()
            ((model(x_train[idx]) - y_train[idx]) ** 2).mean().backward()
            optimizer.step()
    with torch.no_grad():
        err = (torch.linalg.norm(model(x_test) - y_test) / torch.linalg.norm(y_test)).item()
        reconstruction = model(x_test).cpu().numpy()
    return err, reconstruction


# Reconstruct each regime from 3 electrodes three ways -- memoryless (one snapshot), a ridge
# linear map over the history, and SHRED -- all on the honest temporal hold-out.
boundary, panels = {}, {}
for name, field in [("Organized reentry", fhn_reentry()), ("Fibrillation", fibrillation())]:
    seqs, tgts, tr, te, sensors, n_space = split_by_time(field, 3)
    shred_err, recon = train_shred(seqs, tgts, tr, te, 3, n_space)
    ridge_err = linear_reconstruct(seqs.reshape(len(seqs), -1), tgts, tr, te)
    memoryless_err = linear_reconstruct(seqs[:, -1, :], tgts, tr, te)
    boundary[name] = (memoryless_err, ridge_err, shred_err)
    panels[name] = (tgts[te], recon, sensors)
    print(f"  {name:20s}: memoryless {memoryless_err:.3f} | ridge-linear {ridge_err:.3f} | "
          f"SHRED {shred_err:.3f}")
print("  -> history is essential in BOTH; SHRED's nonlinearity only pays off in the chaotic regime.")

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(13.5, 4), constrained_layout=True)
for ax, name in zip(axes[:2], boundary):
    true, recon, sensors = panels[name]
    ax.imshow(true[:400].T, aspect="auto", cmap="inferno")
    for row in sensors:
        ax.axhline(row, color="#39ff14", lw=0.9)
    m, r, s = boundary[name]
    ax.set_title(f"{name} (green = 3 electrodes)\nSHRED {s:.2f}  |  ridge {r:.2f}  |  memoryless {m:.2f}",
                 fontsize=9.5)
    ax.set_xlabel("time (held-out)")
    ax.set_ylabel("tissue position")
methods = ["memoryless", "linear (ridge)", "SHRED"]
xpos = np.arange(len(boundary)); width = 0.26
for i, method in enumerate(methods):
    axes[2].bar(xpos + (i - 1) * width, [boundary[n][i] for n in boundary], width, label=method)
axes[2].set_xticks(xpos)
axes[2].set_xticklabels(["Organized", "Fibrillation"])
axes[2].set_ylabel("held-out rel-L2")
axes[2].set_title("When does the nonlinearity earn its keep?")
axes[2].legend(fontsize=8)
plt.show()

QC note. Read the two regimes together -- the contrast *is* the point. In **both**, a
memoryless single snapshot fails (~0.6): three electrodes at one instant cannot place a
moving wave, so the electrodes' recent **history** is essential. What differs is what the
history buys. For **organized reentry** the motion is *periodic* -- the tissue's state
traces a simple closed loop -- so a plain *linear* decoder over that history already
reconstructs it (~0.20), and SHRED's nonlinear LSTM only ties it (~0.19): the nonlinearity
does no real work. For **fibrillation** the wavelets drift *chaotically* and *advect* --
the state fills a higher-dimensional attractor and moving localized features have a
slowly-decaying linear representation (the *Kolmogorov n-width* obstruction) -- so the
linear decoder collapses (~0.34) while SHRED holds at ~0.02, an order of magnitude better.
That is the honest claim: **nonlinear sensing earns its keep specifically when the dynamics
are chaotic and advection-dominated, not merely when the field moves.** Two limits remain:
these are *stylized, synthetic* fixtures (a real cardiac field is three-dimensional and
noisy), and both are scored on a strict temporal hold-out (a genuinely later episode, not a
leaky random split).

## 5. Learned features for images: a small CNN

Week 5a read blood cells with a **linear** eigen-basis -- the
eigen-cells, a handful of principal images. We close that loop here with the deep-learning
counterpart on the *same* cells: a small **convolutional neural network** (CNN) that learns
its own nonlinear features by gradient descent. A convolution shares its weights across the
image and stacks local-to-global feature detectors -- structure a flat linear model,
eigen-basis or not, cannot represent.

To compare *fairly*, we train the CNN on the **grayscale** cells -- exactly the input Week 5a's
eigen-cells and LDA used -- against a raw-pixel linear baseline on that same input.
Then, separately, we give a second CNN the cells' full **color** to measure how much the stain
adds on top of the nonlinearity.

In [ ]:
import torch.nn as nn
from sklearn.linear_model import LogisticRegression

from ddm4bio.datasets import get_dataset

blood = get_dataset("bloodmnist")
pay = blood.payload
rgb_tr = np.asarray(pay["train_images"], np.float32) / 255.0
rgb_te = np.asarray(pay["test_images"], np.float32) / 255.0
ytr = pay["train_labels"].ravel()
yte = pay["test_labels"].ravel()
print(f"BloodMNIST ({blood.source}): {rgb_tr.shape[0]} train + {rgb_te.shape[0]} test cells, "
      f"{rgb_tr.shape[1]}x{rgb_tr.shape[2]} RGB, {len(np.unique(ytr))} classes")


def train_cnn(x_train, x_test, in_ch):
    # Same small architecture; only the input-channel count changes (1 gray vs 3 color).
    torch.manual_seed(0)
    net = nn.Sequential(
        nn.Conv2d(in_ch, 16, 3, padding=1), nn.BatchNorm2d(16), nn.ReLU(), nn.MaxPool2d(2),
        nn.Conv2d(16, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2),
        nn.Flatten(), nn.Linear(32 * 7 * 7, 64), nn.ReLU(), nn.Linear(64, 8),
    ).to(device)
    xt = torch.tensor(x_train).to(device)
    yt = torch.tensor(ytr).to(device)
    opt = torch.optim.Adam(net.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()
    gen = torch.Generator().manual_seed(0)
    for _ in range(6):
        net.train()
        for idx in torch.randperm(xt.shape[0], generator=gen).split(256):
            opt.zero_grad()
            loss_fn(net(xt[idx]), yt[idx]).backward()
            opt.step()
    net.eval()
    with torch.no_grad():
        return net(torch.tensor(x_test).to(device)).argmax(1).cpu().numpy()


# Grayscale: the like-for-like input Week 5a's eigen-cells and LDA used.
gray_tr = rgb_tr.mean(axis=-1)[:, None, :, :]      # (N, 1, 28, 28)
gray_te = rgb_te.mean(axis=-1)[:, None, :, :]
pred_gray = train_cnn(gray_tr, gray_te, 1)
cnn_gray = float((pred_gray == yte).mean())

import warnings

from sklearn.exceptions import ConvergenceWarning

# Fair baseline: train on the SAME full training set as the CNN (a faster solver keeps it cheap).
with warnings.catch_warnings():
    warnings.simplefilter("ignore", ConvergenceWarning)
    lin = LogisticRegression(max_iter=120).fit(gray_tr.reshape(gray_tr.shape[0], -1), ytr)
lin_gray = float(lin.score(gray_te.reshape(gray_te.shape[0], -1), yte))

# Full color: how much the cells' stain adds on top of the nonlinearity.
cnn_rgb = float((train_cnn(rgb_tr.transpose(0, 3, 1, 2), rgb_te.transpose(0, 3, 1, 2), 3) == yte).mean())

print(f"grayscale linear (raw-pixel) baseline: {lin_gray:.3f}")
print(f"grayscale CNN (fair vs eigen-cells):   {cnn_gray:.3f}")
print(f"full-color CNN (color adds signal):    {cnn_rgb:.3f}")

In [ ]:
import matplotlib.pyplot as plt

names = blood.labels  # authoritative cell-type names from the loader (real BloodMNIST)
show = np.random.default_rng(1).choice(rgb_te.shape[0], 10, replace=False)
fig, axes = plt.subplots(2, 5, figsize=(13, 5.2))
for ax, j in zip(axes.ravel(), show):
    ax.imshow(gray_te[j, 0], cmap="gray")
    ok = pred_gray[j] == yte[j]
    ax.set_title(f"{names[pred_gray[j]]}\n(true: {names[yte[j]]})",
                 fontsize=7.5, color="green" if ok else "red")
    ax.axis("off")
fig.suptitle(f"Grayscale CNN predictions on held-out blood cells (accuracy {cnn_gray:.0%})")
fig;

QC note. Read as a *like-for-like* ladder on the **grayscale** cells -- the exact input Week 5a's
eigen-cells and LDA used: linear eigen-projection ~0.47, supervised LDA ~0.61, a
raw-pixel linear model ~0.67 (trained, like the CNN, on the whole training set), and the CNN ~0.80. The CNN's learned convolutional features --
responding to local shape and texture -- clearly beat every linear model on the *same* input,
which is the honest version of the claim. Giving a CNN the cells' **color** too lifts it to
~0.91: stain color is genuinely informative, and unlike a flat linear basis a convolutional net
can exploit it -- but that extra ~0.10 is *color*, not nonlinearity, so we report it separately
rather than folding it into the comparison with the eigen-cells. The usual caveat still holds:
unlike the linear-autoencoder = PCA equivalence, none of this carries a closed-form guarantee;
it is validated only empirically on this held-out split.

## 6. Interpretation

In [ ]:
from ddm4bio.interpret import interpretation_block, show_interpretation

block = interpretation_block(
    claim=(
        f"A linear autoencoder is exactly PCA (subspaces agreed to "
        f"{principal_angles_deg.max():.2f} degrees); a nonlinear autoencoder beats "
        f"the flat linear latent on a curved manifold ({linear_err:.2f} -> "
        f"{ae_err:.2f} relative-L2 at the same latent dimension); and SHRED reconstructs the "
        f"tissue field from 3 electrodes, but its nonlinearity only earns its keep in the "
        f"*chaotic* regime -- on fibrillation it beats a ridge linear decoder "
        f"({boundary['Fibrillation'][1]:.0%} -> {boundary['Fibrillation'][2]:.0%}) while on "
        f"organized reentry a linear decoder ties it "
        f"({boundary['Organized reentry'][1]:.0%} vs {boundary['Organized reentry'][2]:.0%}). "
        f"That nonlinear autoencoder is also a reusable, invertible map -- it places 300 "
        f"unseen points at {oos_err:.2f} rel-L2 -- while t-SNE offers no such map at all and "
        f"plain UMAP only an approximate, graph-based .transform (not a smoothly learned one)."
    ),
    limitations_list=[
        "The linear=PCA equivalence has a closed-form guarantee (Eckart-Young); the "
        "nonlinear-autoencoder and SHRED results do NOT -- they are empirical and "
        "depend on initialization and training, and must be reported with that caveat.",
        "The fixtures are small and seeded (an S-curve, plus organized and fibrillating excitable-tissue fixtures) so "
        "the lesson runs in seconds on a CPU; real single-cell manifolds and real cardiac or neural "
        "fields are far higher-dimensional and noisier.",
        "SHRED is scored on a strict temporal hold-out (a later block, not a leaky random split); "
        "its nonlinear advantage is regime-specific -- it shows on chaotic, advection-dominated fields, not on simple periodic ones.",
        "t-SNE and UMAP (Section 3b) are visualization embeddings, not smooth learned maps: "
        "t-SNE has no .transform at all (a new point can't be placed without refitting), while "
        "plain UMAP exposes only an approximate, graph-based .transform/.inverse_transform; "
        "neither yields a reconstruction error, and UMAP can fragment a continuum -- so they "
        "are for seeing structure, not measuring it (Parametric UMAP is the smoothly learned "
        "bridge, not covered here).",
        "Reinforcement learning -- a different paradigm from the representation learning here -- is previewed and developed in the "
        "capstone; the genuinely large sequence models -- transformers and "
        "foundation-model embeddings -- sit beyond this course's scope.",
    ],
)
show_interpretation(block)

## Two ways of reasoning, revisited

We opened by naming two intellectual traditions: **deductive, model-driven** reasoning, which begins from known mechanism -- a rate law, a conservation principle, a differential equation -- and deduces what a system must do; and **inductive, data-driven** reasoning, which begins from measurements and lets the patterns, coordinates, and dynamics emerge without committing to a mechanism up front. Eight weeks in, you have worked the whole spectrum between them, and you can now place every method on it.

Toward the model-driven end sit Week 1's least-squares image registration -- solving a known linear system $A\mathbf{x}=\mathbf{b}$ for the transform that aligns two images -- and Week 2's Hill dose-response fit -- a sigmoid whose *functional form* is fixed in advance, giving interpretable parameters (potency, efficacy, steepness) you can reason with even when it is fit phenomenologically to a viability screen. Next come the true hybrids, where the two reasonings meet: Week 7's SINDy recovers an interpretable governing ODE *from* data, and its Kalman filter *fuses* a trusted mechanism with noisy measurement -- the literal "how to combine them" of Learning Outcome #1 -- while Week 4's compressed sensing leans on a known sparse basis (the model) to invert far-undersampled measurements (the data). Then the balance tips inductive: Week 6 lets data propose clusters and biomarkers but keeps model-driven statistics as referee; Weeks 3 and 5 read structure -- a diagnostic boundary, and principal and independent components -- straight from measurements; and Week 8's deep autoencoders and SHRED sit at the far data-driven end -- which itself splits (Section 3b): the autoencoder is data-driven yet *model-producing*, a reusable invertible function you can carry to new cells, while t-SNE and UMAP are data-driven and *purely descriptive*, a one-off picture with no function attached, learning coordinates with no mechanism named at all.

The intro promised that the most interesting work *braids them together*, and Outcome #1 asked you to judge which paradigm fits a problem -- or how to combine them. That judgement is now yours. The **capstone** is where you exercise it deliberately: a designed case study on real warfarin data, in which you fit a mechanistic PK/PD dosing model and then learn a dosing policy on it -- a genuine hybrid, and you defend where on this spectrum each piece sits.


## Exercises

Week 8 has no separate problem set. Your graded work is the **capstone project**:
a designed, multi-step case study that carries **real warfarin PK/PD data** from
raw measurements to a defensible dosing policy -- the course's whole arc in one
project, built around a genuine model-driven <-> data-driven hybrid.

Building on this lesson, you will:

- **Fit the pharmacology (model-driven).** Fit a one-compartment PK model to the
  real warfarin concentrations and a turnover/indirect-response PD model to the
  anticoagulation effect, validating against the drug's known ~1-2.5-day
  elimination half-life.
- **Characterize the population (data-driven).** Reduce and cluster the
  per-patient PK/PD parameters (PCA/ICA + stability) and relate the variability to
  covariates -- turning real inter-patient spread into a calibrated patient model.
- **Learn to dose (the hybrid).** Build a dosing environment from the fitted,
  variability-aware model and recover an optimal dosing policy with the
  value-iteration-then-Q-learning loop from the
  [reinforcement-learning preview](../capstone_rl_preview.ipynb) -- now grounded in a
  real drug.
- **Validate and interpret.** Show the fit recovers known pharmacology and that
  the policy holds anticoagulation in-window better than a fixed dose, and close
  with an `interpretation_block` stating the claim and its limitations -- including
  that this is pedagogical, not clinical, dosing.

Refer to the [capstone project brief](../capstone.md) for the milestones, deliverables, and
grading.